## Load the data

In [1]:
import wandb
import pandas as pd

run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/student/.netrc.
wandb: Currently logged in as: mistral1992 (mistral1992-udacity) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Generate the Profile Report

In [2]:
import ydata_profiling

profile = ydata_profiling.ProfileReport(df)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 30.95it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

### Observations

- **Size:** 20,000 rows, 16 columns, no duplicate rows.
- **Missing values:** 8,261 missing cells (2.6%) in total:
  - `last_review` and `reviews_per_month`: 4123 missing each (20.6%). These are the same rows – listings that never received a review (`number_of_reviews` has exactly 4123 zeros).
  - `name`: 7 missing, `host_name`: 8 missing.
- **Wrong type:** `last_review` is stored as a string (text), not as a datetime.
- **Price outliers:** `price` ranges from 0 (5 listings with price 0) up to 10000 per night, with a mean of ~153. Prices of 0 and the extreme high values are not realistic for the model we want to build.
- Other notes: `minimum_nights` is highly skewed, `availability_365` has 35.9% zeros, `number_of_reviews` is strongly correlated with `reviews_per_month`.

### Decision on missing values

**Do not impute** missing values in this EDA (nor in the cleaning step). Missing values can also appear in new data at inference time, so they must be handled by the inference pipeline. Handling them here would hide the problem from the model and create training/serving skew.

## Cleaning

Decisions:

1. **Price outliers** – after discussing with stakeholders, the accepted nightly price range is inclusive: `$10 <= price <= $350`. Rows outside this range are dropped. `Series.between` is inclusive on both ends by default; pass `inclusive="both"` explicitly to make the intent clear.
2. **`last_review` type** – convert from text to `datetime`. Missing values become `NaT` and are left as they are.

In [3]:
# After discussing with stakeholders, we will restrict nightly prices to between $10 and $350.
# Drop outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()

# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

## Check the result

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19001 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              19001 non-null  int64         
 1   name                            18994 non-null  object        
 2   host_id                         19001 non-null  int64         
 3   host_name                       18993 non-null  object        
 4   neighbourhood_group             19001 non-null  object        
 5   neighbourhood                   19001 non-null  object        
 6   latitude                        19001 non-null  float64       
 7   longitude                       19001 non-null  float64       
 8   room_type                       19001 non-null  object        
 9   price                           19001 non-null  int64         
 10  minimum_nights                  19001 non-null  int64         
 11  number_

`df.info()` after cleaning: 19,001 rows remain (999 rows, ~5%, were removed as price outliers), and `last_review` now has the `datetime64[ns]` type. Missing values are still present (`name`, `host_name`, `last_review`, `reviews_per_month`) – as intended, they are not imputed.

In [5]:
assert df['price'].between(min_price, max_price, inclusive="both").all()
assert pd.api.types.is_datetime64_any_dtype(df['last_review'])

print(f"Rows after cleaning: {len(df)}")
print(f"Price range: {df['price'].min()} - {df['price'].max()}")
print(f"last_review dtype: {df['last_review'].dtype}")
print("\nMissing values (not imputed – handled by the inference pipeline):")
print(df.isna().sum()[df.isna().sum() > 0])

Rows after cleaning: 19001
Price range: 10 - 350
last_review dtype: datetime64[ns]

Missing values (not imputed – handled by the inference pipeline):
name                    7
host_name               8
last_review          3758
reviews_per_month    3758
dtype: int64


Confirmed: 

- all prices are within the defined (inclusive) range,
- `last_review` is now of datetime type.

## Profile the cleaned data

Re-run the profile on the cleaned dataframe to confirm the price distribution is now within `[10, 350]` and `last_review` is detected as a DateTime column.

In [6]:
profile = ydata_profiling.ProfileReport(df)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 31.34it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
# Close the run so the notebook and its results are uploaded to W&B.
run.finish()